# Notebook 18: Training LLMs for Agent Behavior

**Sprint 3: Agentic AI** | Frontier ML Interview Prep Toolkit

---

Current agentic systems rely heavily on prompting: we give an LLM instructions on how to use tools, plan, and recover from errors. But what if we could *train* the model to be a better agent? This is the research frontier, and it connects everything from Sprints 1-3.

**Prerequisites**: Notebooks 5-8 (post-training: SFT, DPO, RLHF, GRPO) and Notebooks 13-17 (agentic patterns)

**Key insight**: Agent training IS post-training (from Sprint 2), applied to agentic behavior (from Sprint 3). If you understand both, you understand the frontier.

---
## 1. Self-Quiz (Active Recall)

**Before reading anything**, try to answer these from memory. Write your answers below.

1. **How do you train an LLM to use tools better?** What does the training data look like?
2. **What is trajectory training?** How do you collect and filter agent trajectories?
3. **How does RLHF apply to agents?** What serves as the reward signal?
4. **What is Agent Q?** What makes it different from simpler approaches?
5. **Why not just prompt a bigger model?** When does training beat prompting?

In [ ]:
# YOUR ANSWERS (write before reading further)
self_quiz_answers = {
    "train_tool_use": "",
    "trajectory_training": "",
    "rlhf_for_agents": "",
    "agent_q": "",
    "train_vs_prompt": "",
}

# After completing the notebook, come back and grade yourself:
# How many did you get right? ___/5

---
## 2. Setup

In [ ]:
!pip install -q torch transformers datasets accelerate

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import json
import random
import time
import math
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Any
from collections import defaultdict

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## 3. Why Train Agents?

### The Current State: Prompted Agents

Today's agents (including production multi-agent RCA systems) work by prompting:
- Give the LLM a system prompt explaining the tools
- Provide few-shot examples of tool use
- Hope the model generalizes to new situations

This works surprisingly well with frontier models (GPT-4, Claude, or an self-hosted model), but has limitations:

| Limitation | Description |
|---|---|
| **Fragile tool calling** | Wrong argument types, hallucinated tool names, wrong tool selection |
| **Poor error recovery** | Model doesn't know what to do when a tool call fails |
| **Prompt sensitivity** | Small prompt changes cause large behavior changes |
| **No learning from experience** | Each conversation starts from scratch |
| **Cost** | Large models are expensive; can't use them for every agent |

### The Promise: Trained Agents

What if we could fine-tune a model to be inherently good at agent tasks?

| Aspect | Prompted Agent | Trained Agent |
|---|---|---|
| Tool calling accuracy | 70-90% (depends on prompt) | 95%+ (learned format) |
| Error recovery | Often gets stuck | Learned recovery patterns |
| Model size needed | Large (GPT-4 class) | Can work with 7B-13B models |
| Cost per query | High | 10-100x lower |
| Latency | High (large model) | Lower (smaller model) |

### The Surprising Result

A 7B model fine-tuned on tool-use trajectories can **outperform** GPT-4 with prompting on specific tool-use benchmarks. This is because:
1. The fine-tuned model has internalized the tool calling format perfectly
2. It has seen thousands of examples of error recovery
3. Its weights encode tool-use patterns, not just its context window

**Interview framing**: "The gap between prompted and trained agents is one of the most exciting research frontiers. It's essentially the post-training stack (SFT, DPO, RLHF) applied to agentic behaviors."

**Insider Tip:** Training agents is the research frontier. Most production agents today are prompted, not trained. The gap between prompted and trained agents is where the next big improvements will come. If asked "what would you work on?", agent training is a strong answer.

---
## 4. Tool Use Fine-Tuning

The most direct approach: collect successful agent trajectories and SFT on them.

In [ ]:
# ============================================================
# Training Data Format for Tool Use
# ============================================================

# A single training example is a complete agent trajectory:
# (system_prompt, user_query, [action_1, observation_1, ..., final_answer])

EXAMPLE_TRAJECTORY = {
    "system_prompt": """You are a helpful assistant with access to the following tools:
- search(query: str) -> str: Search the web for information
- calculate(expression: str) -> float: Evaluate a math expression
- lookup(entity: str) -> dict: Look up information about an entity

Use tools by writing: <tool_call>tool_name(arg1, arg2)</tool_call>
After receiving an observation, continue reasoning.""",
    
    "user_query": "What is the population of France divided by the population of Switzerland?",
    
    "trajectory": [
        {"role": "assistant", "content": "I need to find the populations of both countries.\n<tool_call>lookup(\"France\")</tool_call>"},
        {"role": "tool", "content": "{\"name\": \"France\", \"population\": 67390000, \"capital\": \"Paris\"}"},
        {"role": "assistant", "content": "France has about 67.39 million people. Now Switzerland.\n<tool_call>lookup(\"Switzerland\")</tool_call>"},
        {"role": "tool", "content": "{\"name\": \"Switzerland\", \"population\": 8776000, \"capital\": \"Bern\"}"},
        {"role": "assistant", "content": "Now I can calculate the ratio.\n<tool_call>calculate(\"67390000 / 8776000\")</tool_call>"},
        {"role": "tool", "content": "7.68"},
        {"role": "assistant", "content": "The population of France (67.39 million) divided by the population of Switzerland (8.78 million) is approximately **7.68**. France has roughly 7.7 times the population of Switzerland."}
    ],
    
    "success": True,
    "num_tool_calls": 3,
    "num_steps": 4  # 3 tool calls + 1 final answer
}

print("Example trajectory:")
print(f"  Query: {EXAMPLE_TRAJECTORY['user_query']}")
print(f"  Steps: {EXAMPLE_TRAJECTORY['num_steps']}")
print(f"  Tool calls: {EXAMPLE_TRAJECTORY['num_tool_calls']}")
print(f"  Success: {EXAMPLE_TRAJECTORY['success']}")

In [ ]:
# ============================================================
# ToolUseDataset: Format trajectories for SFT
# ============================================================

class ToolUseDataset(Dataset):
    """Dataset of agent trajectories formatted for supervised fine-tuning."""
    
    def __init__(self, trajectories: List[Dict], tokenizer=None, max_length: int = 512):
        self.trajectories = trajectories
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.formatted = [self._format_trajectory(t) for t in trajectories]
        # Parallel segment view used for loss masking: list of (text, train_on) pairs.
        self.segments = [self._format_segments(t) for t in trajectories]
    
    def _format_trajectory(self, traj: Dict) -> str:
        """Convert a trajectory dict into a training string."""
        parts = [f"<|system|>{traj['system_prompt']}<|end|>"]
        parts.append(f"<|user|>{traj['user_query']}<|end|>")
        
        for step in traj["trajectory"]:
            if step["role"] == "assistant":
                parts.append(f"<|assistant|>{step['content']}<|end|>")
            elif step["role"] == "tool":
                parts.append(f"<|tool_result|>{step['content']}<|end|>")
        
        return "\n".join(parts)
    
    def _format_segments(self, traj: Dict) -> List[Tuple[str, bool]]:
        """Like _format_trajectory, but returns (text, train_on) segments.
        
        train_on=True only for assistant turns. FireAct/AgentTuning-style agent
        SFT masks the prompt AND the tool/observation (environment-generated)
        tokens with -100 so the model learns to PRODUCE actions, not to
        imitate the environment's outputs.
        """
        segs = [(f"<|system|>{traj['system_prompt']}<|end|>", False)]
        segs.append((f"\n<|user|>{traj['user_query']}<|end|>", False))
        for step in traj["trajectory"]:
            if step["role"] == "assistant":
                segs.append((f"\n<|assistant|>{step['content']}<|end|>", True))
            elif step["role"] == "tool":
                segs.append((f"\n<|tool_result|>{step['content']}<|end|>", False))
        return segs
    
    def __len__(self):
        return len(self.formatted)
    
    def __getitem__(self, idx):
        if self.tokenizer is not None:
            # Tokenize segment-by-segment so non-assistant spans (system/user prompt
            # and tool results) can be masked with -100: they contribute no loss.
            input_ids, labels = [], []
            for seg_text, train_on in self.segments[idx]:
                seg_ids = self.tokenizer(seg_text, add_special_tokens=False)["input_ids"]
                input_ids.extend(seg_ids)
                labels.extend(seg_ids if train_on else [-100] * len(seg_ids))
            input_ids = input_ids[:self.max_length]
            labels = labels[:self.max_length]
            attention_mask = [1] * len(input_ids)
            pad_len = self.max_length - len(input_ids)
            input_ids += [self.tokenizer.pad_token_id] * pad_len
            labels += [-100] * pad_len
            attention_mask += [0] * pad_len
            return {
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
            }
        
        return {"text": self.formatted[idx]}


# ============================================================
# Generate Synthetic Training Data
# ============================================================

def generate_synthetic_trajectories(n: int = 50) -> List[Dict]:
    """Generate synthetic tool-use trajectories for training demo."""
    
    system_prompt = """You are a helpful assistant with tools:
- search(query: str) -> str: Search for information
- calculate(expr: str) -> float: Evaluate math
- lookup(entity: str) -> dict: Entity lookup
Use: <tool_call>tool_name(args)</tool_call>"""
    
    templates = [
        {
            "query": "What is the GDP of {country}?",
            "trajectory": [
                {"role": "assistant", "content": "Let me look that up.\n<tool_call>lookup(\"{country}\")</tool_call>"},
                {"role": "tool", "content": '{{"name": "{country}", "gdp_billions": {gdp}}}'},
                {"role": "assistant", "content": "The GDP of {country} is approximately ${gdp} billion."},
            ],
            "params": [
                {"country": "Germany", "gdp": 4256},
                {"country": "Japan", "gdp": 4231},
                {"country": "Brazil", "gdp": 1920},
                {"country": "India", "gdp": 3737},
                {"country": "Canada", "gdp": 2140},
            ]
        },
        {
            "query": "What is {a} * {b} + {c}?",
            "trajectory": [
                {"role": "assistant", "content": "Let me calculate that.\n<tool_call>calculate(\"{a} * {b} + {c}\")</tool_call>"},
                {"role": "tool", "content": "{result}"},
                {"role": "assistant", "content": "{a} * {b} + {c} = {result}"},
            ],
            "params": [
                {"a": 15, "b": 23, "c": 7, "result": 352},
                {"a": 42, "b": 8, "c": 100, "result": 436},
                {"a": 99, "b": 11, "c": 55, "result": 1144},
                {"a": 7, "b": 13, "c": 29, "result": 120},
                {"a": 256, "b": 4, "c": 32, "result": 1056},
            ]
        },
        {
            "query": "Search for recent news about {topic}",
            "trajectory": [
                {"role": "assistant", "content": "I'll search for that.\n<tool_call>search(\"{topic} recent news\")</tool_call>"},
                {"role": "tool", "content": "Top result: {news_headline}"},
                {"role": "assistant", "content": "Here's what I found about {topic}: {news_headline}"},
            ],
            "params": [
                {"topic": "quantum computing", "news_headline": "Google achieves new quantum supremacy milestone"},
                {"topic": "Mars exploration", "news_headline": "NASA's Perseverance rover discovers organic molecules"},
                {"topic": "AI regulation", "news_headline": "EU AI Act enters enforcement phase"},
                {"topic": "renewable energy", "news_headline": "Solar energy costs drop below coal globally"},
                {"topic": "CRISPR", "news_headline": "New CRISPR variant enables precise single-base editing"},
            ]
        },
    ]
    
    trajectories = []
    for _ in range(n):
        template = random.choice(templates)
        params = random.choice(template["params"])
        
        query = template["query"].format(**params)
        traj_steps = []
        for step in template["trajectory"]:
            traj_steps.append({
                "role": step["role"],
                "content": step["content"].format(**params)
            })
        
        trajectories.append({
            "system_prompt": system_prompt,
            "user_query": query,
            "trajectory": traj_steps,
            "success": True,
            "num_tool_calls": sum(1 for s in traj_steps if s["role"] == "tool"),
            "num_steps": len(traj_steps),
        })
    
    return trajectories


# Generate training data
train_trajectories = generate_synthetic_trajectories(50)
dataset = ToolUseDataset(train_trajectories)

print(f"Generated {len(train_trajectories)} training trajectories")
print(f"\nExample formatted trajectory:")
print(dataset[0]["text"][:500])

In [ ]:
# ============================================================
# SFT on Tool Use Traces (Simplified Demo with GPT-2)
# ============================================================

from transformers import GPT2LMHeadModel, GPT2Tokenizer

print("Loading GPT-2 for tool-use fine-tuning demo...")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Create dataset with tokenizer
train_dataset = ToolUseDataset(train_trajectories, tokenizer=tokenizer, max_length=256)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

# Training loop (abbreviated for demo)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
model.train()

num_epochs = 2
losses = []

for epoch in range(num_epochs):
    epoch_loss = 0
    num_batches = 0
    
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        # Labels already mask the prompt, tool/observation, and padding tokens
        # with -100 (see ToolUseDataset): FireAct/AgentTuning-style agent SFT
        # trains only on assistant tokens so the model doesn't learn to imitate
        # environment-generated outputs.
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
    
    avg_loss = epoch_loss / num_batches
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")

print(f"\nTraining complete. Loss went from {losses[0]:.4f} to {losses[-1]:.4f}")
print("In a real setting, you'd train on 1000+ trajectories for 3-5 epochs.")

In [ ]:
# ============================================================
# Before/After: Test Tool Calling Format
# ============================================================

def test_tool_calling(model, tokenizer, prompt: str, max_new_tokens: int = 100) -> str:
    """Generate a response and check if it follows tool calling format."""
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


test_prompt = """<|system|>You are a helpful assistant with tools:
- search(query: str) -> str: Search for information
- calculate(expr: str) -> float: Evaluate math
- lookup(entity: str) -> dict: Entity lookup
Use: <tool_call>tool_name(args)</tool_call><|end|>
<|user|>What is the population of Japan?<|end|>
<|assistant|>"""

response = test_tool_calling(model, tokenizer, test_prompt)
print("Model output:")
print(response[-200:])  # Show the end of the generation

# Check for tool calling format
has_tool_call = "<tool_call>" in response
print(f"\nContains tool call format: {has_tool_call}")
print("\nNote: With only 50 examples and 2 epochs on GPT-2, this is a minimal demo.")
print("In production, you'd use a 7B+ model with 1000+ examples.")

---
## 5. Trajectory Training

The key insight: collect many agent trajectories, filter by quality, and train on the best ones. This is **rejection sampling** applied to agent behavior.

In [ ]:
# ============================================================
# TrajectoryCollector: Run agent on tasks, collect traces
# ============================================================

@dataclass
class Trajectory:
    """A complete agent trajectory with outcome."""
    task: str
    steps: List[Dict]  # [{"action": ..., "observation": ...}, ...]
    final_answer: str
    success: bool
    reward: float  # 0.0 to 1.0
    num_steps: int = 0
    total_tokens: int = 0
    latency_seconds: float = 0.0
    
    def __post_init__(self):
        self.num_steps = len(self.steps)
    
    @property
    def efficiency(self) -> float:
        """Lower is better: steps taken relative to success."""
        if not self.success:
            return float('inf')
        return self.num_steps


class TrajectoryCollector:
    """Collect agent trajectories by running an agent on many tasks."""
    
    def __init__(self):
        self.trajectories: List[Trajectory] = []
    
    def collect(self, tasks: List[Dict], runs_per_task: int = 3) -> List[Trajectory]:
        """Run the agent on each task multiple times, collect all trajectories."""
        all_trajectories = []
        
        for task_info in tasks:
            task = task_info["query"]
            expected = task_info.get("expected_answer", None)
            
            for run_id in range(runs_per_task):
                # Simulate running an agent (in production, this calls the actual agent)
                traj = self._simulate_agent_run(task, expected)
                all_trajectories.append(traj)
        
        self.trajectories.extend(all_trajectories)
        return all_trajectories
    
    def _simulate_agent_run(self, task: str, expected: Optional[str]) -> Trajectory:
        """Simulate an agent run with variable quality (for demo)."""
        # Simulate different quality levels
        quality = random.random()
        num_steps = random.randint(1, 6)
        
        steps = []
        for i in range(num_steps):
            steps.append({
                "action": f"tool_call_{i}",
                "observation": f"result_{i}",
                "reasoning": f"Step {i} reasoning"
            })
        
        success = quality > 0.3  # 70% success rate
        reward = quality if success else quality * 0.2
        
        # Better trajectories tend to have fewer, more targeted steps
        if success and num_steps <= 3:
            reward = min(1.0, reward + 0.2)  # Bonus for efficiency
        
        return Trajectory(
            task=task,
            steps=steps,
            final_answer=f"Answer for: {task}" if success else "Failed to answer",
            success=success,
            reward=reward,
            total_tokens=random.randint(200, 2000),
            latency_seconds=random.uniform(0.5, 5.0)
        )


# Collect trajectories
collector = TrajectoryCollector()

tasks = [
    {"query": "What is the capital of France?", "expected_answer": "Paris"},
    {"query": "Calculate 17 * 23 + 5", "expected_answer": "396"},
    {"query": "Search for recent AI news", "expected_answer": None},
    {"query": "Look up the population of Germany", "expected_answer": "84 million"},
    {"query": "What is 2^10?", "expected_answer": "1024"},
    {"query": "Search for RLHF vs DPO papers", "expected_answer": None},
    {"query": "What is the GDP of Japan?", "expected_answer": "$4.2 trillion"},
    {"query": "Calculate the square root of 144", "expected_answer": "12"},
]

trajectories = collector.collect(tasks, runs_per_task=4)

print(f"Collected {len(trajectories)} trajectories")
print(f"Success rate: {sum(t.success for t in trajectories) / len(trajectories):.1%}")
print(f"Avg reward: {sum(t.reward for t in trajectories) / len(trajectories):.3f}")
print(f"Avg steps: {sum(t.num_steps for t in trajectories) / len(trajectories):.1f}")

In [ ]:
# ============================================================
# TrajectoryFilter: Select best trajectories for training
# ============================================================

class TrajectoryFilter:
    """Filter and rank trajectories by quality for training data curation."""
    
    def __init__(self, trajectories: List[Trajectory]):
        self.trajectories = trajectories
    
    def filter_successful(self) -> List[Trajectory]:
        """Keep only successful trajectories."""
        return [t for t in self.trajectories if t.success]
    
    def rank_by_reward(self) -> List[Trajectory]:
        """Sort by reward (highest first)."""
        return sorted(self.trajectories, key=lambda t: t.reward, reverse=True)
    
    def rank_by_efficiency(self) -> List[Trajectory]:
        """Sort by efficiency (fewest steps first, among successful)."""
        successful = self.filter_successful()
        return sorted(successful, key=lambda t: (t.num_steps, -t.reward))
    
    def select_top_k(self, k: int, method: str = "reward") -> List[Trajectory]:
        """Select top-k trajectories by specified method."""
        if method == "reward":
            ranked = self.rank_by_reward()
        elif method == "efficiency":
            ranked = self.rank_by_efficiency()
        else:
            ranked = self.filter_successful()
        
        return ranked[:k]
    
    def create_preference_pairs(self) -> List[Tuple[Trajectory, Trajectory]]:
        """Create (chosen, rejected) pairs for DPO training.
        
        For each task, pair the best trajectory (chosen) with a worse one (rejected).
        This is the key insight: we can turn trajectory data into preference data.
        """
        # Group by task
        task_groups = defaultdict(list)
        for t in self.trajectories:
            task_groups[t.task].append(t)
        
        pairs = []
        for task, trajs in task_groups.items():
            if len(trajs) < 2:
                continue
            
            # Sort by reward
            sorted_trajs = sorted(trajs, key=lambda t: t.reward, reverse=True)
            
            # Pair best with worst
            chosen = sorted_trajs[0]
            rejected = sorted_trajs[-1]
            
            if chosen.reward > rejected.reward:  # Only if there's a clear difference
                pairs.append((chosen, rejected))
        
        return pairs


# Apply filtering
filter = TrajectoryFilter(trajectories)

successful = filter.filter_successful()
top_10 = filter.select_top_k(10, method="reward")
efficient = filter.select_top_k(10, method="efficiency")
preference_pairs = filter.create_preference_pairs()

print(f"Total trajectories: {len(trajectories)}")
print(f"Successful: {len(successful)}")
print(f"Top-10 by reward: avg reward = {sum(t.reward for t in top_10)/len(top_10):.3f}")
print(f"Top-10 by efficiency: avg steps = {sum(t.num_steps for t in efficient)/len(efficient):.1f}")
print(f"Preference pairs for DPO: {len(preference_pairs)}")

print("\nPreference pair example:")
if preference_pairs:
    chosen, rejected = preference_pairs[0]
    print(f"  Task: {chosen.task}")
    print(f"  Chosen:   reward={chosen.reward:.3f}, steps={chosen.num_steps}, success={chosen.success}")
    print(f"  Rejected: reward={rejected.reward:.3f}, steps={rejected.num_steps}, success={rejected.success}")

### Connection to Sprint 2

This IS post-training, applied to agent behavior:

| Sprint 2 Concept | Agent Training Application |
|---|---|
| **SFT** (Notebook 5) | Fine-tune on successful agent trajectories |
| **DPO** (Notebook 6) | Use (good trajectory, bad trajectory) as preference pairs |
| **Rejection Sampling** (Notebook 7) | Generate many trajectories, keep only the best |
| **GRPO** (Notebook 8) | Generate G trajectories per task, reward = task completion |

The only difference is the training data format: instead of (prompt, response) pairs, you have (prompt, [action, observation, ...]) trajectories.

---
## 6. RLHF for Agents

Apply reinforcement learning to optimize agent behavior. The reward signal comes from task completion, not human preferences.

In [ ]:
# ============================================================
# AgentGRPO: Group Relative Policy Optimization for Agents
# ============================================================
# This is GRPO from Notebook 8 / DeepSeek-R1, applied to agent tool use.

class AgentGRPO:
    """GRPO training for agent behavior.
    
    For each task:
    1. Generate G agent trajectories (with different random seeds)
    2. Score each trajectory (reward = task completion + efficiency)
    3. Compute group-normalized advantages
    4. Update policy with weighted log probabilities
    
    This is a simplified GRPO-style update applied to tool use (the real GRPO
    adds the clipped importance ratio and a KL term to the reference policy).
    """
    
    def __init__(self, model, tokenizer, G: int = 4, lr: float = 1e-5):
        self.model = model
        self.tokenizer = tokenizer
        self.G = G  # Group size: trajectories per task
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
        self.training_log = []
    
    def generate_trajectories(self, task: str) -> List[Dict]:
        """Generate G different agent trajectories for the same task."""
        trajectories = []
        
        for g in range(self.G):
            # Each trajectory uses different temperature / sampling
            trajectory = self._simulate_trajectory(task, run_id=g)
            trajectories.append(trajectory)
        
        return trajectories
    
    def _simulate_trajectory(self, task: str, run_id: int) -> Dict:
        """Simulate an agent trajectory (replace with actual agent in production)."""
        quality = random.random()
        num_steps = random.randint(1, 5)
        success = quality > 0.25  # 75% success rate
        
        return {
            "task": task,
            "run_id": run_id,
            "steps": [{"action": f"step_{i}"} for i in range(num_steps)],
            "success": success,
            "num_steps": num_steps,
            "quality": quality,
        }
    
    def compute_rewards(self, trajectories: List[Dict]) -> List[float]:
        """Score each trajectory.
        
        Reward components:
        - Task completion: 1.0 if success, 0.0 if not
        - Efficiency bonus: shorter trajectories get a bonus
        - Quality: how good was the final answer
        """
        rewards = []
        for traj in trajectories:
            reward = 0.0
            
            # Task completion (binary)
            if traj["success"]:
                reward += 1.0
            
            # Efficiency bonus (fewer steps = better)
            if traj["success"]:
                efficiency_bonus = max(0, 1.0 - traj["num_steps"] / 5.0) * 0.3
                reward += efficiency_bonus
            
            # Quality component
            reward += traj["quality"] * 0.2
            
            rewards.append(reward)
        
        return rewards
    
    def compute_advantages(self, rewards: List[float]) -> List[float]:
        """Group-normalize rewards to get advantages.
        
        This is the key GRPO insight: normalize within the group.
        Advantage_i = (reward_i - mean(rewards)) / std(rewards)
        """
        mean_reward = sum(rewards) / len(rewards)
        std_reward = (sum((r - mean_reward) ** 2 for r in rewards) / len(rewards)) ** 0.5
        
        if std_reward < 1e-8:  # All rewards are the same
            return [0.0] * len(rewards)
        
        advantages = [(r - mean_reward) / std_reward for r in rewards]
        return advantages
    
    def update(self, trajectories: List[Dict], advantages: List[float]) -> float:
        """Policy gradient update weighted by advantages.
        
        In a full implementation, this would:
        1. Compute log P(trajectory | model) for each trajectory
        2. Multiply by the advantage
        3. Gradient ascent on the weighted log probabilities
        
        Here we demonstrate the math with a simplified version.
        """
        self.model.train()
        
        total_loss = 0.0
        for traj, advantage in zip(trajectories, advantages):
            if abs(advantage) < 1e-8:
                continue  # Skip zero-advantage trajectories
            
            # Simplified: create a training text from the trajectory
            text = f"Task: {traj['task']} Steps: {traj['num_steps']} Success: {traj['success']}"
            inputs = self.tokenizer(text, return_tensors="pt", max_length=128, 
                                   truncation=True, padding="max_length").to(device)
            
            labels = inputs["input_ids"].clone()
            labels[labels == self.tokenizer.pad_token_id] = -100
            
            outputs = self.model(**inputs, labels=labels)
            
            # Policy gradient: maximize E[advantage * log_prob]
            # Since outputs.loss = -mean(log_prob) (NLL), we have:
            #   log_prob ≈ -outputs.loss
            # To maximize advantage * log_prob with gradient DESCENT:
            #   minimize -(advantage * log_prob) = advantage * outputs.loss
            # Positive advantage -> minimize NLL (reinforce behavior)
            # Negative advantage -> maximize NLL (discourage behavior)
            weighted_loss = advantage * outputs.loss
            
            self.optimizer.zero_grad()
            weighted_loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.optimizer.step()
            
            total_loss += weighted_loss.item()
        
        return total_loss / max(len(trajectories), 1)
    
    def train_step(self, task: str) -> Dict:
        """Complete GRPO training step for one task."""
        # 1. Generate G trajectories
        trajectories = self.generate_trajectories(task)
        
        # 2. Compute rewards
        rewards = self.compute_rewards(trajectories)
        
        # 3. Compute advantages (group normalization)
        advantages = self.compute_advantages(rewards)
        
        # 4. Update policy
        loss = self.update(trajectories, advantages)
        
        result = {
            "task": task,
            "rewards": rewards,
            "advantages": advantages,
            "mean_reward": sum(rewards) / len(rewards),
            "max_reward": max(rewards),
            "loss": loss,
            "success_rate": sum(1 for t in trajectories if t["success"]) / len(trajectories)
        }
        
        self.training_log.append(result)
        return result


# Run GRPO training
grpo = AgentGRPO(model, tokenizer, G=4, lr=1e-5)

training_tasks = [
    "Find the population of France",
    "Calculate 25 * 17",
    "Search for recent AI breakthroughs",
    "Look up the capital of Brazil",
    "What is the square root of 256?"
]

print("GRPO Training for Agent Tool Use")
print("=" * 50)

for i, task in enumerate(training_tasks):
    result = grpo.train_step(task)
    print(f"\nTask {i+1}: {task}")
    print(f"  Rewards: {[f'{r:.3f}' for r in result['rewards']]}")
    print(f"  Advantages: {[f'{a:.3f}' for a in result['advantages']]}")
    print(f"  Mean reward: {result['mean_reward']:.3f}")
    print(f"  Success rate: {result['success_rate']:.0%}")

# Summary
avg_reward = sum(r["mean_reward"] for r in grpo.training_log) / len(grpo.training_log)
avg_success = sum(r["success_rate"] for r in grpo.training_log) / len(grpo.training_log)
print(f"\n{'='*50}")
print(f"Overall: avg reward = {avg_reward:.3f}, avg success = {avg_success:.0%}")
print(f"\nThis is a simplified GRPO-style update (the real GRPO adds the clipped")
print(f"importance ratio and a KL term to the reference policy), applied to tool use.")

---
## 7. Agent Q: MCTS + DPO

Agent Q (2024) is one of the most sophisticated approaches to agent training. It combines Monte Carlo Tree Search (MCTS) with DPO. Signature result: on real-world web booking (OpenTable), it took Llama-3-70B from an 18.6% zero-shot success rate to 81.7% after a single day of autonomous data collection, and to 95.4% with online MCTS.

### Key Idea

Instead of just collecting complete trajectories and filtering, Agent Q explores the **tree of possible actions** at each step:

```
                    Task: "Find population of France"
                              |
                    Step 1: What tool to call?
                    /         |         \\
            search()    lookup()    calculate()
              |              |             |
          [rollout]    [rollout]      [rollout]
          reward=0.3   reward=0.9    reward=0.1
```

At each decision point:
1. **Expand**: Consider multiple possible tool calls
2. **Rollout**: Complete each trajectory to the end, measure reward
3. **Select**: The best action (by rollout reward) becomes the "chosen" action
4. **Train**: Use MCTS-selected actions as "chosen" and random/worse actions as "rejected" for DPO

### Why This Is Better Than Simple Trajectory Filtering

| Approach | Data Quality | Data Efficiency | Compute Cost |
|---|---|---|---|
| **Rejection sampling** | Medium (whole trajectories) | Low (many wasted runs) | Low |
| **Best-of-N** | Good (top trajectories) | Medium | Medium |
| **Agent Q (MCTS + DPO)** | Excellent (step-level) | High (every rollout informs training) | High |

### The MCTS Advantage

- **Step-level preference data**: Not just "this trajectory was good", but "at step 3, calling `lookup()` was better than calling `search()`"
- **Exploration**: MCTS explores actions the base model would never try, discovering better strategies
- **Credit assignment**: Rewards are attributed to specific decisions, not the whole trajectory

### Algorithm Sketch

```
for each training task:
    root = initial state (task description)
    
    for each step in trajectory:
        # Expand: generate K possible actions
        actions = model.generate(state, K=5, temperature=1.0)
        
        # Rollout: complete each action to get final reward
        for action in actions:
            reward = complete_trajectory_and_evaluate(state + action)
        
        # Select best action
        best_action = max(actions, key=lambda a: a.reward)
        worst_action = min(actions, key=lambda a: a.reward)
        
        # Create DPO pair
        dpo_pairs.append((best_action, worst_action))
        
        # Advance state
        state = state + best_action
    
    # Train with DPO on collected pairs
    dpo_train(model, dpo_pairs)
```

### Why It Matters (Interview)

Agent Q represents the cutting edge (2024-2025) of agent training:
- It connects **planning** (MCTS from AlphaGo) with **language model training** (DPO)
- It shows that the same techniques that solved Go can improve LLM agents
- It's computationally expensive but produces the best-trained agents

> "Agent Q uses MCTS to generate step-level preference data for DPO training. At each decision point in an agent trajectory, it explores multiple possible tool calls, evaluates them via rollout, and uses the best/worst pairs for DPO. This gives much richer training signal than trajectory-level filtering because it attributes credit to individual decisions."

---
## 8. Distillation: Strong Agent to Weak Agent

Use a strong model (GPT-4, Claude) to generate agent traces, then train a small model to imitate them.

In [ ]:
# ============================================================
# Agent Distillation Pipeline
# ============================================================

class AgentDistillation:
    """Distill a strong agent's behavior into a smaller model.
    
    Pipeline:
    1. Run strong agent (GPT-4) on many tasks -> collect traces
    2. Filter successful traces
    3. Format as training data
    4. SFT small model on these traces
    5. Evaluate: how close does the small model get?
    """
    
    def __init__(self, strong_model_name: str = "gpt-4", weak_model_name: str = "gpt-2"):
        self.strong_model_name = strong_model_name
        self.weak_model_name = weak_model_name
        self.traces: List[Dict] = []
    
    def collect_strong_traces(self, tasks: List[str], traces_per_task: int = 3) -> List[Dict]:
        """Collect agent traces from the strong model.
        
        In production: this calls GPT-4 with tool use enabled.
        Here: we simulate high-quality traces.
        """
        traces = []
        
        for task in tasks:
            for _ in range(traces_per_task):
                trace = self._simulate_strong_trace(task)
                traces.append(trace)
        
        self.traces = traces
        return traces
    
    def _simulate_strong_trace(self, task: str) -> Dict:
        """Simulate a high-quality trace from a strong model."""
        # Strong models have higher success rate and better efficiency
        success = random.random() > 0.1  # 90% success rate
        num_steps = random.randint(1, 3)  # Strong models are efficient
        
        steps = []
        for i in range(num_steps):
            steps.append({
                "thought": f"I need to {['search', 'calculate', 'look up'][i % 3]} for this.",
                "action": f"tool_call('{task}')",
                "observation": f"Result for step {i}"
            })
        
        return {
            "task": task,
            "steps": steps,
            "answer": f"Answer: {task}" if success else "Failed",
            "success": success,
            "model": self.strong_model_name,
            "quality_score": random.uniform(0.7, 1.0) if success else random.uniform(0.0, 0.3)
        }
    
    def filter_and_format(self, min_quality: float = 0.5) -> List[Dict]:
        """Filter high-quality traces and format for SFT."""
        filtered = [t for t in self.traces if t["success"] and t["quality_score"] >= min_quality]
        
        formatted = []
        for trace in filtered:
            text_parts = [f"Task: {trace['task']}"]
            for i, step in enumerate(trace["steps"]):
                text_parts.append(f"Thought: {step['thought']}")
                text_parts.append(f"Action: {step['action']}")
                text_parts.append(f"Observation: {step['observation']}")
            text_parts.append(f"Answer: {trace['answer']}")
            
            formatted.append({
                "text": "\n".join(text_parts),
                "quality_score": trace["quality_score"]
            })
        
        return formatted
    
    def evaluate_distillation(self, strong_results: List[Dict], 
                               weak_results: List[Dict]) -> Dict:
        """Compare strong model vs distilled weak model."""
        strong_success = sum(1 for r in strong_results if r["success"]) / len(strong_results)
        weak_success = sum(1 for r in weak_results if r["success"]) / len(weak_results)
        
        strong_steps = sum(r.get("num_steps", 0) for r in strong_results) / len(strong_results)
        weak_steps = sum(r.get("num_steps", 0) for r in weak_results) / len(weak_results)
        
        return {
            "strong_model": {
                "success_rate": strong_success,
                "avg_steps": strong_steps,
            },
            "weak_model_distilled": {
                "success_rate": weak_success,
                "avg_steps": weak_steps,
            },
            "gap": strong_success - weak_success,
            "retention_rate": weak_success / max(strong_success, 1e-8),
        }


# Run the distillation pipeline
distiller = AgentDistillation(strong_model_name="gpt-4o", weak_model_name="llama-7b")

# Step 1: Collect strong traces
tasks = [
    "What is the capital of Japan?",
    "Calculate 15% of 840",
    "Search for the latest SpaceX launch",
    "Look up the GDP of Germany",
    "What is 2^15?",
    "Find the distance from Earth to Mars",
    "Calculate the area of a circle with radius 7",
    "Search for Nobel Prize winners 2024",
]

traces = distiller.collect_strong_traces(tasks, traces_per_task=3)
print(f"Collected {len(traces)} traces from strong model")
print(f"Success rate: {sum(t['success'] for t in traces)/len(traces):.1%}")

# Step 2: Filter and format
training_data = distiller.filter_and_format(min_quality=0.5)
print(f"\nFiltered to {len(training_data)} high-quality training examples")

# Step 3: Evaluate (simulated)
# In reality, you'd train the weak model and evaluate on held-out tasks
strong_results = [{"success": random.random() > 0.1, "num_steps": random.randint(1,3)} for _ in range(50)]
weak_results = [{"success": random.random() > 0.25, "num_steps": random.randint(1,4)} for _ in range(50)]

eval_results = distiller.evaluate_distillation(strong_results, weak_results)
print(f"\nDistillation Results:")
print(f"  Strong model ({distiller.strong_model_name}):")
print(f"    Success rate: {eval_results['strong_model']['success_rate']:.1%}")
print(f"    Avg steps: {eval_results['strong_model']['avg_steps']:.1f}")
print(f"  Weak model ({distiller.weak_model_name}), distilled:")
print(f"    Success rate: {eval_results['weak_model_distilled']['success_rate']:.1%}")
print(f"    Avg steps: {eval_results['weak_model_distilled']['avg_steps']:.1f}")
print(f"  Retention rate: {eval_results['retention_rate']:.1%} of strong model performance")
print(f"\nKey insight: The distilled 7B model retains ~{eval_results['retention_rate']:.0%} of GPT-4's")
print(f"agent performance at 1/100th the cost per query.")

### Why Distillation Works for Agents

1. **Format is learnable**: Tool calling syntax is a pattern that small models can memorize
2. **Reasoning is compressible**: The strong model's chain-of-thought can be distilled into direct action
3. **Specialization helps**: A 7B model fine-tuned for one tool set can beat GPT-4 for that specific use case
4. **The 80/20 rule**: 80% of agent tasks are routine; only 20% need frontier-model reasoning

### Production Strategy

```
User Query
    |
    v
[Router / Complexity Classifier]
    |
    |-- Simple (80%) --> Distilled 7B Agent (fast, cheap)
    |
    |-- Complex (20%) --> GPT-4 Agent (slow, expensive)
```

This is a common pattern at frontier labs: use a small specialized model for most queries, escalate to a large model for hard ones.

---
## 9. The Full Picture: Post-Training for Agents

Everything from Sprints 1-3 connects here. This is the synthesis.

### The Training Pipeline

```
Pre-trained LLM (Sprint 1: Foundation)
        |
        v
SFT on tool-use trajectories (Sprint 1-2: Supervised Fine-Tuning)
  - Train on successful agent traces
  - Model learns tool calling format, basic planning
        |
        v
DPO on trajectory preferences (Sprint 2: Direct Preference Optimization)
  - (good trajectory, bad trajectory) pairs
  - Model learns to prefer successful, efficient strategies
        |
        v
GRPO on verifiable agent tasks (Sprint 2: Group Relative Policy Optimization)
  - Generate G trajectories per task
  - Reward = task completion (verifiable!)
  - Group normalization -> advantages -> policy gradient
  - This is DeepSeek-R1's approach for reasoning, applied to tool use
        |
        v
Agent Q: MCTS + DPO (Cutting edge)
  - Explore action space at each step with MCTS
  - Step-level preference data for DPO
  - Best training signal but most expensive
        |
        v
Deployed Agent (Sprint 3: Agentic Patterns)
  - ReAct loop, tool use, memory, planning
  - Multi-agent orchestration
  - The trained model is much more reliable than a prompted one
```

### Why This Is the Research Frontier

Most frontier AI labs (OpenAI, Anthropic, Google DeepMind, Meta) are actively working on agent training because:

1. **Agents are the product**: Claude, ChatGPT, Gemini are all becoming agentic
2. **Prompted agents hit a ceiling**: Prompting alone can't make agents reliable enough for production
3. **RL for agents is tractable**: Unlike open-ended RLHF, agent tasks have verifiable rewards (did the tool call work?)
4. **Small trained agents can beat large prompted ones**: This enables cost-effective deployment

### Interview Synthesis

> "Agent training connects everything in the modern ML stack. You start with a pre-trained LLM, apply SFT to teach it the tool calling format, DPO to optimize for successful trajectories, and GRPO to train with verifiable task completion rewards. The cutting edge -- Agent Q -- uses MCTS to explore the action space at each step and generates step-level preference data for DPO.
>
> This matters because prompted agents are fragile and expensive. A fine-tuned 7B model can match GPT-4's agent performance on specific tasks at 1/100th the cost. In production, you'd use a router: small trained models for routine tasks, frontier models for complex ones.
>
> These principles apply directly to a multi-agent RCA (root-cause-analysis) system, where each of three agents is optimized for its specific role -- the same post-training approach applied to planning, orchestration, and analysis behaviors."

---
## 10. "Why Does This Work?" -- Deep Questions

### Q: Why not just prompt a bigger model?

Three reasons:

1. **Cost**: GPT-4 cost ~$30/1M input tokens (2024-era pricing; check current rates). A fine-tuned 7B model on your own GPU costs ~$0.10/1M tokens. At scale, this is the difference between viable and not.

2. **Latency**: Larger models are slower. For real-time agent applications (interactive assistants, monitoring systems), you need fast responses. A 7B model generates tokens 10-20x faster than GPT-4.

3. **Reliability**: A prompted model can "forget" its instructions mid-trajectory. A fine-tuned model has the tool calling pattern in its weights, not just its context. This makes it much more consistent.

### Q: How much training data do you need?

Surprisingly little for tool use:

| Data Volume | What You Get |
|---|---|
| **50-100 examples** | Model learns the tool calling format reliably |
| **100-500 examples** | Model learns when to use which tool |
| **500-1000 examples** | Model learns multi-step planning and error recovery |
| **1000-5000 examples** | Model generalizes to new tools and task types |

This is because tool use is highly structured. The format is repetitive (thought -> action -> observation), so a few hundred examples provide strong signal.

### Q: What's the limit of training vs prompting?

This is an open research question. Current understanding:

- **Training wins** for: format adherence, speed, cost, consistency
- **Prompting wins** for: flexibility, zero-shot generalization, rapid iteration
- **The frontier**: Can we get the best of both? Train a base agent model, then prompt it for specific tasks? This is the approach most frontier labs are converging on.

---
## Interview Question Bank

*Agent training questions are the most research-oriented in the Sprint 3 interview loop. These are open-ended discussions where the interviewer wants to see how you think about unsolved problems.*

---

### Q1: "How would you train an LLM to be a better agent?" -- RESEARCH DISCUSSION (30 min)

**What this tests**: This is an OPEN-ENDED research question. There is no single right answer. The interviewer is evaluating your research breadth, ability to reason about trade-offs, and awareness of what is unsolved.

**Good answer** (hire):
- SFT on successful agent trajectories: collect traces of successful task completions, fine-tune the model to imitate them
- Use DPO or RLHF to prefer successful trajectories over failed ones

**Great answer** (strong hire): Discusses the full spectrum of approaches and their trade-offs:

1. **SFT on trajectories**: The baseline. Collect successful agent traces, format as (system_prompt, user_query, [action_1, observation_1, action_2, observation_2, ..., final_answer]). Fine-tune. Simple but limited -- the model only learns to imitate, not to reason about why actions were chosen.

2. **DPO on trajectory pairs**: Collect (successful_trace, failed_trace) pairs for the same task. Train the model to prefer the successful one. Better than pure SFT because the model learns from contrast. But requires paired data, which is expensive to collect.

3. **GRPO with task completion reward**: Use Group Relative Policy Optimization (from DeepSeek-R1) with a reward based on task success/failure. The model generates multiple trajectories, successful ones get higher reward. No paired data needed. But the reward signal is sparse (only at the end of the trajectory).

4. **MCTS for data collection (Agent Q)**: Use Monte Carlo Tree Search to explore the action space, find successful paths through difficult tasks, then train on those paths. This generates high-quality training data for tasks where random exploration rarely succeeds. State of the art as of 2024-2025.

5. **Process reward models**: Train a reward model that evaluates each step of the trajectory, not just the final outcome. This provides denser reward signal and allows credit assignment (which step was the mistake?).

**Connects to Sprint 2**: "This is essentially the same post-training pipeline we covered in Sprint 2 (SFT -> preference optimization -> RL), but applied to trajectories instead of single responses. The key difference is that agent trajectories are much longer and the reward is delayed."

**What is unsolved**: "The fundamental challenge is credit assignment -- when an agent fails after 20 steps, which step was the mistake? Was it the plan, the tool choice, the argument formatting, or the interpretation of the result? Dense reward models help but are themselves hard to train."

**Red flag**: Only mentions SFT. Does not discuss RL-based approaches or what is unsolved.

**Follow-up 1**: "What is the right reward signal for training a general-purpose agent?"
- Good: task completion (binary)
- Great: "There probably is not a single right reward signal. I would use a combination: (1) task completion as the primary signal, (2) efficiency as a secondary signal (fewer steps is better), (3) safety as a hard constraint (never reward unsafe actions regardless of task completion). The weighting depends on the deployment context. For a coding agent, correctness dominates. For a customer service agent, safety and tone might be equally important."

**Follow-up 2**: "How much training data do you need for reliable tool use?"
- Good: "Thousands of examples"
- Great: "It depends on the tool complexity. For simple tools (calculator, search), 100-500 examples of correct tool calls is often sufficient -- the format is learnable. For complex tools (APIs with many parameters, conditional logic), you need 1,000-5,000 examples. For tool SELECTION (choosing the right tool from 20+), you need 5,000-10,000 examples covering the selection space. Toolformer showed that self-supervised generation can bootstrap this data, but the quality is lower than human-annotated traces."

---

### Q2: "Compare training vs prompting for agent behavior"

**What this tests**: Practical engineering judgment about when to invest in training vs when prompting is sufficient.

**Good answer**:
- Training is more reliable but less flexible (requires new training for new behaviors)
- Prompting is more flexible but less reliable (depends on in-context learning)

**Great answer**: Discusses the Pareto frontier across three axes:

| Axis | Prompting | Training | Distillation |
|------|----------|----------|-------------|
| **Cost per query** | High (big model needed) | Low (small model works) | Low (small model) |
| **Upfront cost** | Zero | $1K-100K | $100-10K |
| **Flexibility** | Very high (change prompt) | Low (need retraining) | Low (need retraining) |
| **Reliability** | Medium | High | Medium-High |
| **Time to deploy** | Minutes | Days-weeks | Hours-days |
| **Best for** | Prototyping, rare tools | High-frequency, reliability-critical | Cost optimization at scale |

**When prompting is sufficient**:
- Prototyping and iteration (you are still figuring out what the agent should do)
- Rare tool use (tool is called <100 times/day -- not worth training for)
- Rapidly changing tools (API changes frequently -- retraining is wasteful)

**When training is necessary**:
- High-frequency, reliability-critical tool use (>10,000 calls/day where 99%+ accuracy matters)
- Cost optimization at scale (prompted GPT-4 at $10/1M tokens vs fine-tuned Llama-70B at $1/1M tokens -- 10x cheaper)
- Specific behavioral patterns that prompting cannot reliably produce (complex multi-turn tool orchestration)

**The distillation approach**: Use a large prompted model (GPT-4, Claude Opus) to generate high-quality agent traces, then fine-tune a small model (Llama-8B, Mistral-7B) on those traces. This captures the large model's behavior at a fraction of the cost. Many production systems use this approach.

**Red flag**: Says "always train" or "always prompt." Does not consider the trade-off space.

---
## Production Implementation Notes

*The state of agent training in production -- what works today and what is still research.*

### The Current State of Agent Training (2025)

**Most production agents TODAY are prompted, not trained.** Training for agent behavior is cutting-edge research. Here is the landscape:

| Approach | Who Uses It | Maturity | Cost |
|----------|-----------|----------|------|
| **Prompted large model** (GPT-4, Claude Opus) | Everyone (default) | Production-ready | $10-30/1M tokens |
| **Prompted small model** (GPT-4o-mini, Haiku) | Cost-sensitive applications | Production-ready | $0.15-1/1M tokens |
| **SFT on trajectories** | Companies with proprietary data | Early production | $1K-10K training + $1/1M inference |
| **DPO on trajectory pairs** | Research labs | Research/experimental | $5K-50K training |
| **GRPO/RL from task success** | Frontier labs only | Research | $50K-500K training |
| **MCTS + training (Agent Q)** | Published research only | Pre-production | $100K+ |

### Key Research Results

**Toolformer (Schick et al., 2023)**:
- Showed that self-supervised tool learning is possible: the model teaches itself when and how to use tools by comparing perplexity with and without tool calls
- Limitation: only works for simple tools (calculator, search, calendar). Does not scale to complex APIs.
- Significance: proved the concept that models can learn tool use without human-annotated tool call examples

**Agent Q (2024 -- MCTS + DPO)**:
- Represents the state of the art for trained agents
- Uses MCTS to explore the action space and find successful trajectories through difficult tasks
- Then trains with DPO on (successful, failed) trajectory pairs
- Result: on OpenTable web booking, Llama-3-70B went from 18.6% to 81.7% success after one day of autonomous data collection, and 95.4% with online MCTS
- Limitation: MCTS exploration is very expensive (thousands of rollouts per task)

**ATLaS (2025, arXiv 2503.02197)**:
- Key insight: not all steps in a trajectory are equally important for training
- Uses "critical step selection" to identify the decisions that matter most and weight them higher during training
- This addresses the credit assignment problem partially -- focus training on the steps where the agent's choice made the difference between success and failure

### The Gap That Matters for Interviews

**Prompted GPT-4 vs fine-tuned Llama-70B for agent tasks**: The fine-tuned smaller model often wins at 1/10th the inference cost.

This is a crucial data point for interviews because it challenges the "just use the biggest model" assumption. When an interviewer asks "how would you build an agent for X?", mentioning distillation as a cost optimization strategy shows production thinking.

The workflow:
1. Prototype with prompted GPT-4/Claude (fast iteration, no training cost)
2. Collect successful trajectories from production (data flywheel)
3. Fine-tune Llama-70B on those trajectories (10x cost reduction)
4. Use the fine-tuned model for production, keep the large model for edge cases (escalation pattern)

### What is NOT Solved

- **Generalizable agent training**: Models trained on one tool set do not generalize well to new tools. You essentially need to retrain for each new tool configuration.
- **Safety during training**: RL-trained agents can discover unsafe strategies that are technically "successful" (e.g., exploiting API bugs, manipulating user feedback). Constraining the training process is an open problem.
- **Sample efficiency**: Current RL approaches need thousands of trajectories. Humans can learn to use a new tool from 2-3 examples.
- **Credit assignment**: When a 20-step trajectory fails, which step was wrong? Process reward models help but are themselves hard to build.

---
## How This Gets Tested in Interviews

### Agent Training Is the "Research Depth" Question

Agent training questions distinguish ML researchers from ML engineers. An ML engineer knows how to prompt an agent. A ML researcher knows how to make the underlying model better at being an agent. This is the highest-signal question in the Sprint 3 topic area.

### What the Interviewer Is Looking For

1. **Breadth across the training spectrum**: Can you discuss SFT, DPO, GRPO, MCTS, and process reward models fluently? Not just names -- actual trade-offs, when to use each, what data is needed.

2. **Connection to post-training fundamentals**: Agent training IS post-training (Sprint 2) applied to trajectories. If you can explain how DPO on chat responses (Sprint 2) extends to DPO on agent trajectories (Sprint 3), you demonstrate deep understanding of the underlying principles.

3. **Awareness of what is unsolved**: The strongest signal in a research interview is when a candidate can clearly articulate what we do NOT know how to do yet. "Credit assignment in long trajectories is unsolved" and "RL-trained agents can discover unsafe strategies" are the kinds of statements that get people hired.

4. **Cost-aware reasoning**: "We could train with MCTS + DPO for $100K, or we could prompt GPT-4 for $0.01/query. The break-even point is 10 million queries. For our use case with 100K queries/month, prompting is cheaper for the first 8 years. So we should prompt." This is the kind of analysis that separates principal engineers from researchers who have never shipped.

### The Research Discussion Format

Agent training questions typically follow this pattern:

**Phase 1 -- Breadth (5 min)**: "How would you train an agent?" Survey the landscape. Show you know the options.

**Phase 2 -- Depth (10 min)**: Interviewer picks one approach and goes deep. "You mentioned MCTS for trajectory collection. Walk me through exactly how that works." Be ready to explain the algorithm step by step, not just the high-level idea.

**Phase 3 -- Open problems (10 min)**: "What is the hardest unsolved problem in agent training?" This is where you show research taste. Good answers:
- Credit assignment in long trajectories
- Reward specification for general-purpose agents
- Safety constraints during RL training
- Sample efficiency (humans learn tools from examples, models need thousands of trajectories)
- Generalization across tool configurations

**Phase 4 -- Proposal (5 min)**: "If you joined our team, what would you work on first?" Have a concrete answer. "I would focus on process reward models for agent trajectories because credit assignment is the bottleneck, and recent work on PRMs for math (from your team's own research) suggests this approach can work for agents too."

### The Key Narrative for This Topic

When discussing agent training in any interview, anchor on this narrative:

> "Agent training is where post-training meets tool use. We have the building blocks -- SFT, DPO, RL -- from language model training. The challenge is adapting them to the agent setting where trajectories are long, rewards are delayed, actions have real-world consequences, and the tool landscape changes faster than we can retrain. The field is moving from prompted agents (2023-2024) to trained agents (2025+), and whoever solves the credit assignment and safety problems will define the next generation of AI systems."

This positions you as someone who understands both the current state and the trajectory of the field -- exactly what a frontier lab wants in a senior research hire.

---
## 11. Flashcard Summary

| # | Question | Answer |
|---|---|---|
| 1 | What is tool-use fine-tuning? | SFT on successful agent trajectories: (system_prompt, user_query, [action, observation, ..., answer]) |
| 2 | What is trajectory training? | Collect many agent trajectories, filter by quality (rejection sampling), train on the best ones |
| 3 | How do you create preference pairs from trajectories? | For each task, pair the highest-reward trajectory (chosen) with the lowest-reward one (rejected) for DPO |
| 4 | How does GRPO apply to agents? | Generate G trajectories per task, reward = task completion, group-normalize, policy gradient update. Same as DeepSeek-R1 but for tool use. |
| 5 | What is Agent Q? | MCTS + DPO: explore multiple actions at each step via rollout, use best/worst as DPO pairs. Step-level preference data. |
| 6 | Why is Agent Q better than trajectory filtering? | Step-level credit assignment (not trajectory-level), explores actions the model wouldn't normally try, richer training signal |
| 7 | What is agent distillation? | Use a strong model (GPT-4) to generate agent traces, then SFT a small model on those traces |
| 8 | How much data for tool-use fine-tuning? | 100-1000 examples is often enough. Tool use is highly structured so the model learns quickly. |
| 9 | Why train agents instead of prompting bigger models? | Cost (100x cheaper), latency (10-20x faster), reliability (patterns in weights, not context) |
| 10 | How does agent training connect to Sprint 2? | It IS Sprint 2: SFT teaches format, DPO optimizes trajectories, GRPO uses verifiable rewards, all applied to agentic behavior |
| 11 | What is the agent training pipeline? | Pre-train -> SFT (tool format) -> DPO (trajectory preferences) -> GRPO (verifiable rewards) -> Deploy with agentic patterns |
| 12 | When does a small trained model beat a large prompted one? | On specific, well-defined tool-use tasks where the small model was fine-tuned on that exact domain |

---
## 12. Paper Guides

### Paper 1: Toolformer (Schick et al., 2023)

**Link**: https://arxiv.org/abs/2302.04761

**Key Takeaways**:
- Trains a language model to decide *when and how* to use external tools (calculator, search, etc.)
- Self-supervised: the model generates tool calls, filters by whether they improve predictions, trains on the filtered data
- No human annotation needed for tool use training data
- Showed that even a 6.7B model can learn effective tool use

**What to focus on for interviews**:
- Section 3: The self-supervised data generation pipeline (this is the key innovation)
- The filtering criterion: a tool call is "good" if it reduces the model's loss on the next token
- Table 2: Results showing tool-augmented model beats much larger models on factual tasks

**Critical evaluation**: Toolformer only learns to use tools within a single forward pass. It doesn't learn multi-step planning or error recovery. But it proved the concept: LLMs can learn tool use from self-supervised data.

---

### Paper 2: FireAct (Chen et al., 2023)

**Link**: https://arxiv.org/abs/2310.05915

**Key Takeaways**:
- Fine-tunes LLMs on agent trajectories generated by GPT-4 (distillation approach)
- Compares ReAct, Chain-of-Thought, and reflexion prompting strategies as training data
- Key finding: fine-tuned Llama-2 13B matches GPT-4 prompting on some agent benchmarks
- Training on diverse prompting strategies (multi-task) gives the best results

**What to focus on for interviews**:
- Table 1: The comparison between prompted GPT-4 and fine-tuned smaller models
- Section 4: Training methodology (how to format trajectories for SFT)
- The insight that multi-strategy training data beats single-strategy

---

### Paper 3: Agent Q (Putta et al., 2024)

**Link**: https://arxiv.org/abs/2408.07199

**Key Takeaways**:
- Combines MCTS with DPO for agent training
- Uses MCTS to explore the action space at each step, generating step-level preference data
- Achieves state-of-the-art on WebShop and other agent benchmarks
- Shows that search-based data generation dramatically improves training efficiency

**What to focus on for interviews**:
- The connection between MCTS (from game-playing AI) and language model training
- How step-level DPO pairs are constructed from MCTS rollouts
- The computational cost vs. data quality tradeoff

**Critical evaluation**: Agent Q is computationally expensive (many rollouts per training example). But it produces the highest-quality training data, and compute is getting cheaper. This approach may become standard.

---

### Bonus Reading

- **ToolBench** (Qin et al., 2023): Large-scale benchmark and training data for tool use
- **Gorilla** (Patil et al., 2023): LLM trained specifically for API calling
- **TRICE** (Qiao et al.): "Making Language Models Better Tool Learners with Execution Feedback" -- trains tool use with execution feedback
- **ETO** (Song et al., 2024): "Trial and Error: Exploration-Based Trajectory Optimization" -- learns agents from exploration failures

### Additional Paper: ATLaS (2025, arXiv 2503.02197) -- Training LLMs for Tool Use Through Critical Step Selection

**Key Idea**: Not all steps in an agent trajectory are equally important for learning. ATLaS identifies "critical steps" -- the moments where the agent makes a decision that determines success or failure -- and focuses training on those steps.

**Why It Matters**:
- Standard SFT on full trajectories wastes compute on trivial steps (e.g., formatting, repeating instructions)
- By weighting loss on critical steps, training is more sample-efficient
- Represents a broader trend: curating training data quality matters more than quantity for agent training

**Interview Relevance**: If asked "how would you improve agent fine-tuning?", mentioning step-level importance weighting (rather than uniform loss over the full trajectory) shows sophisticated understanding of the training data pipeline.